### Tools

- Tools extend what agents can do—letting them fetch real-time data, execute code, query external databases, and take actions in the world.
- Under the hood, tools are callable functions with well-defined inputs and outputs that get passed to a chat model. The model decides when to invoke a tool based on the conversation context, and what input arguments to provide.

In [ ]:
# !pip uninstall langchian langchain-openai langchain-community -y
# !pip install langchain langchain-openai langchain-community

In [ ]:
from langchain.tools import tool

### Basic Tool Definition
- The simplest way to create a tool is with the **@tool** decorator.
- By default, the function’s docstring becomes the tool’s description that helps the model understand when to use it
- Type hints are required as they define the tool’s input schema.
- The docstring should be informative and concise to help the model understand the tool’s purpose.
- By default, the tool name comes from the function name. Override it when you need something more descriptive
- Override the auto-generated tool description for clearer model guidance


In [ ]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

### Agents
- Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.
- **create_agent** provides a production-ready agent implementation.
- An LLM Agent runs tools in a loop to achieve a goal. An agent runs until a stop condition is met - i.e., when the model emits a final output or an iteration limit is reached.









In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

# Creating an agent with the defined tools
agent = create_agent(model, tools=[search, get_weather])

In [ ]:
agent.invoke(<query>)

- Tools give agents the ability to take actions. 
- Agents go beyond simple model-only tool binding by facilitating:
    - Multiple tool calls in sequence (triggered by a single prompt)
    - Parallel tool calls when appropriate
    - Dynamic tool selection based on previous results
    - Tool retry logic and error handling
    - State persistence across tool calls

### Example #1

- pip uninstall langchain langchain-openai langchain-community -y
- pip install "langchain>=0.1.0" langchain-openai langchain-community

##### Read the API Keys

In [9]:
key_file = r"C:\mindful-ai\samsung\purushotham-ai\key-vault\groq\api.key"

with open(key_file, "r", encoding="utf-8") as f:
    groq_api_key = f.read().strip()

In [ ]:
#f = open(r"E:\Lenovo Ideapad 330\company-material\ai-upskill\key-vault\openweather\openweather-api-key.txt")
openweather_apikey = r""
#f.close()

In [3]:
city = "Bengaluru"
!curl "http://api.openweathermap.org/data/2.5/weather?q=${city}&appid=${openweather_apikey}&units=metric"

{"cod":401, "message": "Invalid API key. Please see https://openweathermap.org/faq#error401 for more info."}


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100    108 100    108   0      0    562      0                              0
100    108 100    108   0      0    562      0                              0
100    108 100    108   0      0    561      0                              0


##### Define the calculator tools

In [4]:
from langchain.tools import tool

@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calculator(expression: str) -> str:
    """Evaluate mathematical expressions."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"

##### Define the weather tools

In [5]:
import requests
from langchain.tools import tool

@tool("get_weather", description="Get weather information for a location.")
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    api_key = openweather_apikey
    url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={api_key}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        weather_desc = data['weather'][0]['description']
        temp = data['main']['temp']
        return f"Weather in {location}: {weather_desc}, {temp}°C"
    else:
        return f"Error fetching weather data for {location}."
    

##### Build the agent

In [ ]:
!pip install langchain-groq

In [6]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from groq import Groq
from langchain_groq import ChatGroq

In [10]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # or another supported Groq model
    api_key=groq_api_key,
    temperature=0,
)

tools = [calculator, get_weather]

agent = create_agent(llm, tools)

##### Invoke the agent

In [11]:
# Invoke agent to check a calculation
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the weather in Bangalore"}
    ]
})
print(response)

{'messages': [HumanMessage(content='What is the weather in Bangalore', additional_kwargs={}, response_metadata={}, id='44f80e0b-d707-4b7c-98a5-d2a9d9a1652a'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '397q72v37', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 277, 'total_tokens': 292, 'completion_time': 0.037055983, 'completion_tokens_details': None, 'prompt_time': 0.014673509, 'prompt_tokens_details': None, 'queue_time': 0.163749473, 'total_time': 0.051729492}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fff31-6124-77b1-acb6-a8ba1894bd1e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '397q72v37', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

In [12]:
result = response["messages"][-1].content
print(result)

The current weather in Bangalore is overcast clouds with a temperature of 27.37°C.


In [15]:
# Invoke agent to check calculator
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 25 * 4 + 10?"}
    ]
})
print(response)

{'messages': [HumanMessage(content='What is 25 * 4 + 10?', additional_kwargs={}, response_metadata={}, id='ed5ac442-8651-4753-91f2-653df78ac571'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ynpjzyv4y', 'function': {'arguments': '{"expression":"25 * 4 + 10"}', 'name': 'calculator'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 282, 'total_tokens': 302, 'completion_time': 0.025883851, 'completion_tokens_details': None, 'prompt_time': 0.018706468, 'prompt_tokens_details': None, 'queue_time': 0.049705715, 'total_time': 0.044590319}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fff33-15e0-78e3-9348-83860f759524-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 4 + 10'}, 'id': 'ynpjzyv4y', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

In [ ]:
result = response["messages"][-1].content
print(result)